# U-Net and ViT depth comparison
In Colab, choose a GPU runtime and run all cells. Supply the original vit_depth_best.pt when prompted. The notebook trains U-Net and evaluates both checkpoints on the original drive-level validation split. The last cell downloads the trained U-Net, a metrics CSV, and paired validation images for Streamlit.

In [ ]:
from pathlib import Path
import math
import re

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = Path("data")
CHECKPOINT = Path("unet_depth_best.pt")
HEIGHT, WIDTH = 256, 512
SEED = 42
MAX_EPOCHS, PATIENCE = 40, 5
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
from pathlib import Path
import subprocess
import sys

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not any(DATA_ROOT.glob("**/proj_depth/groundtruth/image_02/*.png")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run([
        "kaggle", "datasets", "download", "-d",
        "aalniak/kitti-eigen-split-for-monocular-depth-estimation",
        "--unzip", "-p", str(DATA_ROOT),
    ], check=True)

rgb_count = sum(
    p.suffix.lower() in {".png", ".jpg", ".jpeg"}
    for p in DATA_ROOT.glob("**/image_02/data/*")
)
depth_count = sum(1 for _ in DATA_ROOT.glob("**/proj_depth/groundtruth/image_02/*.png"))
print(f"RGB files: {rgb_count} | Depth files: {depth_count}")
if not rgb_count or not depth_count:
    raise FileNotFoundError(f"Expected both RGB and depth files under {DATA_ROOT}.")

In [ ]:
def find_pairs(root):
    drive_pattern = re.compile(r"\d{4}_\d{2}_\d{2}_drive_\d+_sync")

    def index_frames(paths, prefer_png=False):
        frames = {}
        for path in sorted(paths):
            drive = next((p.name for p in path.parents if drive_pattern.fullmatch(p.name)), None)
            if drive is None:
                continue
            key = (drive, path.stem)
            previous = frames.get(key)
            if previous is None:
                frames[key] = path
            elif prefer_png and previous.parent == path.parent and previous.suffix.lower() != path.suffix.lower():
                if path.suffix.lower() == ".png":
                    frames[key] = path
            else:
                raise ValueError(f"Duplicate frame {key}: {previous} and {path}")
        return frames

    rgb = index_frames(
        (p for p in root.glob("**/image_02/data/*")
         if p.suffix.lower() in {".png", ".jpg", ".jpeg"}),
        prefer_png=True,
    )
    depth = index_frames(root.glob("**/proj_depth/groundtruth/image_02/*.png"))
    return [(rgb[key], depth[key], key[0]) for key in sorted(rgb.keys() & depth.keys())]


def read_rgb(path):
    with Image.open(path) as image:
        pixels = np.asarray(
            image.convert("RGB").resize((WIDTH, HEIGHT), Image.Resampling.BILINEAR),
            dtype=np.float32,
        )
    return torch.from_numpy(pixels).permute(2, 0, 1) / 255.0


class KITTIDepth(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        rgb_path, depth_path, _ = self.pairs[index]
        with Image.open(depth_path) as depth:
            depth = depth.resize((WIDTH, HEIGHT), Image.Resampling.NEAREST)
            meters = np.asarray(depth, dtype=np.float32) / 256.0
        return read_rgb(rgb_path), torch.from_numpy(meters).unsqueeze(0)


pairs = find_pairs(DATA_ROOT)
drives = [drive for _, _, drive in pairs]
if len(set(drives)) < 2:
    raise ValueError(f"Found {len(pairs)} pairs in {DATA_ROOT.resolve()}. Extract KITTI with at least two paired drives.")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(pairs, groups=drives))
train_ds = KITTIDepth([pairs[i] for i in train_idx])
val_ds = KITTIDepth([pairs[i] for i in val_idx])
print(f"Matched {len(pairs)} pairs | Train: {len(train_ds)} | Validation: {len(val_ds)}")
print(f"Validation drives: {sorted({drives[i] for i in val_idx})}")

batch_size = 4 if device.type == "cuda" else 1
loader_options = dict(batch_size=batch_size, pin_memory=device.type == "cuda")
train_loader = DataLoader(train_ds, shuffle=True, **loader_options)
val_loader = DataLoader(val_ds, **loader_options)

In [ ]:
class ViTDepth(nn.Module):
    def __init__(self):
        super().__init__()
        patch_size, dim = 16, 128
        self.grid = (HEIGHT // patch_size, WIDTH // patch_size)
        self.patch_embedding = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        self.position = nn.Parameter(torch.empty(1, math.prod(self.grid), dim))
        nn.init.trunc_normal_(self.position, std=0.02)
        block = nn.TransformerEncoderLayer(
            dim, nhead=4, dim_feedforward=4 * dim, dropout=0.1,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            block, num_layers=4, norm=nn.LayerNorm(dim), enable_nested_tensor=False,
        )
        self.depth_head = nn.Sequential(
            nn.Conv2d(dim, 64, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.GELU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, rgb):
        patches = self.patch_embedding(rgb)
        tokens = patches.flatten(2).transpose(1, 2)
        features = self.encoder(tokens + self.position).transpose(1, 2)
        features = features.reshape(rgb.shape[0], -1, *self.grid)
        depth = self.depth_head(features)
        return F.softplus(F.interpolate(
            depth, size=rgb.shape[-2:], mode="bilinear", align_corners=False,
        ))

class UNetDepth(nn.Module):
    def __init__(self):
        super().__init__()

        def block(inputs, outputs):
            return nn.Sequential(
                nn.Conv2d(inputs, outputs, 3, padding=1), nn.ReLU(),
                nn.Conv2d(outputs, outputs, 3, padding=1), nn.ReLU(),
            )

        self.pool = nn.MaxPool2d(2)
        self.enc1 = block(3, 16)
        self.enc2 = block(16, 32)
        self.enc3 = block(32, 64)
        self.bridge = block(64, 128)
        self.dec3 = block(128 + 64, 64)
        self.dec2 = block(64 + 32, 32)
        self.dec1 = block(32 + 16, 16)
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, rgb):
        a = self.enc1(rgb)
        b = self.enc2(self.pool(a))
        c = self.enc3(self.pool(b))
        value = self.bridge(self.pool(c))
        for skip, decode in ((c, self.dec3), (b, self.dec2), (a, self.dec1)):
            value = F.interpolate(value, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            value = decode(torch.cat((value, skip), dim=1))
        return F.softplus(self.head(value))

In [ ]:
from google.colab import files
from io import BytesIO
vit_path = Path("/content/vit_depth_best.pt")
if not vit_path.exists():
    print("Choose your original vit_depth_best.pt")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select one ViT checkpoint.")
    vit_weights = BytesIO(next(iter(uploaded.values())))
else:
    vit_weights = vit_path
vit = ViTDepth().to(device)
vit.load_state_dict(torch.load(vit_weights, map_location=device, weights_only=True))
vit.eval()
torch.manual_seed(SEED)
model = UNetDepth().to(device)

## Train U-Net
Use masked MAE, AdamW, the same learning rate, batch-size rule, epoch limit and early-stopping patience as the original ViT. Compare runs on the same runtime type.

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    abs_error = squared_error = 0.0
    valid_pixels = 0
    for rgb, target in loader:
        valid = target > 0
        count = int(valid.sum())
        if count == 0:
            continue
        rgb, target, valid = rgb.to(device), target.to(device), valid.to(device)
        with torch.set_grad_enabled(training):
            error = model(rgb)[valid] - target[valid]
            absolute = error.abs().sum()
            loss = absolute / count
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        abs_error += absolute.detach().item()
        squared_error += error.detach().square().sum().item()
        valid_pixels += count
    if valid_pixels == 0:
        raise ValueError("This split has no valid depth pixels.")
    return abs_error / valid_pixels, math.sqrt(squared_error / valid_pixels)


optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
best_mae, best_epoch, stale_epochs = float("inf"), 0, 0
for epoch in range(1, MAX_EPOCHS + 1):
    train_mae, _ = run_epoch(model, train_loader, optimizer)
    val_mae, val_rmse = run_epoch(model, val_loader)
    if not all(map(math.isfinite, (train_mae, val_mae, val_rmse))):
        raise FloatingPointError(f"Non-finite metric at epoch {epoch}.")
    print(f"Epoch {epoch:02d} | Train MAE {train_mae:.3f} m | "
          f"Val MAE {val_mae:.3f} m | Val RMSE {val_rmse:.3f} m")
    if val_mae < best_mae:
        best_mae, best_epoch, stale_epochs = val_mae, epoch, 0
        torch.save(model.state_dict(), CHECKPOINT)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(torch.load(CHECKPOINT, map_location=device, weights_only=True))
model.eval()
print(f"Restored epoch {best_epoch} ({best_mae:.3f} m validation MAE).")

In [ ]:
import csv
import statistics
import time

@torch.inference_mode()
def score(network):
    network.eval()
    totals = torch.zeros(4, 3, dtype=torch.float64, device=device)
    ranges = [(0, float("inf")), (0, 10), (10, 30), (30, float("inf"))]
    for rgb, target in val_loader:
        target = target.to(device)
        err = (network(rgb.to(device)) - target).double()
        for row, (low, high) in enumerate(ranges):
            selected = err[(target > 0) & (target >= low) & (target < high)]
            totals[row, 0] += selected.numel()
            totals[row, 1] += selected.abs().sum()
            totals[row, 2] += selected.square().sum()
    return totals.cpu().tolist()

@torch.inference_mode()
def latency(network, sample):
    times = []
    for step in range(25):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        network(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        if step >= 5:
            times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times)

rows = []
sample = val_ds[0][0].unsqueeze(0).to(device)
for name, network in [("ViT", vit), ("U-Net", model)]:
    scores = score(network)
    ms = latency(network, sample)
    for band, (count, absolute, squared) in zip(["All", "0–<10", "10–<30", "30+"], scores):
        row = dict(model=name, range_m=band, pixels=int(count),
                   mae_m=absolute/count if count else None,
                   rmse_m=math.sqrt(squared/count) if count else None,
                   median_forward_ms=ms, device=str(device))
        rows.append(row)
        print(row)

with open("validation_metrics.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
from zipfile import ZipFile

with ZipFile("depth_comparison.zip", "w") as bundle:
    bundle.write(CHECKPOINT, "unet_depth_best.pt")
    bundle.write("validation_metrics.csv")
    for idx, (rgb_path, _, drive) in enumerate(val_ds.pairs):
        rgb, lidar = val_ds[idx]
        stem = f"{drive}_{rgb_path.stem}"
        photo = Image.fromarray((rgb.permute(1, 2, 0).numpy()*255).round().astype("uint8"))
        target = Image.fromarray((lidar[0].numpy()*256).round().astype("uint16"))
        for folder, image in [("rgb", photo), ("lidar", target)]:
            buf = BytesIO()
            image.save(buf, format="PNG")
            bundle.writestr(f"validation/{folder}/{stem}.png", buf.getvalue())
files.download("depth_comparison.zip")